# Transformers

This example ports trained transformers of scikit-learn (e.g. for the
preprocessing) with [sklearn-porter](https://github.com/nok/sklearn-porter).
A ported transformer has the method `transform`, which returns the
transformed features of a sample, analogous to `predict` of a ported
estimator. The functions `port`, `save`, `make` and `test` accept
transformers too.

Transforming features with the ported transformer requires a compiler or
interpreter of the target language, here `node` for JavaScript.

## Train a transformer

Load the data and train a transformer:

In [1]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
scaler = StandardScaler()
scaler.fit(X)

StandardScaler()

## Port the transformer

**Step 1**: Port (transpile) the transformer and get the source code:

In [2]:
from sklearn_porter import Transformer

transformer = Transformer(scaler, language='js', template='attached')
output = transformer.port()

print(output)

/*
This file is generated by https://github.com/nok/sklearn-porter/

Transformer:
    StandardScaler
    https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

Environment:
    scikit-learn v1.9.1
    sklearn-porter v1.0.0

Usage:
    1. Execute a transformation:
        $ node StandardScaler.js feature_1 ... feature_4
    2. Execute the transformations of several samples (one per line):
        $ node StandardScaler.js < samples.txt
*/
var StandardScaler = function(mean, scale) {

    this.mean = mean;
    this.scale = scale;

    this.transform = function(features) {
        var values = new Array(this.mean.length);
        for (var i = 0, l = this.mean.length; i < l; i++) {
            values[i] = (features[i] - this.mean[i]) / this.scale[i];
        }
        return values;
    };

};

// Parse a feature, also `nan`, `inf` and `-inf` (e.g. of Python) and `NaN`,
// `Infinity` and `-Infinity` (e.g. of a ported transformer):
var parseFeature = fun

**Step 2**: Save the ported transformer:

In [3]:
src_path = transformer.save(directory='/tmp')

print(src_path)

/tmp/StandardScaler.js


**Step 3**: Transform the features with the ported transformer:

In [4]:
X_tr = transformer.make(X[::30])

print(X_tr)

[[-9.00681170e-01  1.01900435e+00 -1.34022653e+00 -1.31544430e+00]
 [-1.26418478e+00  9.82172869e-02 -1.22655167e+00 -1.31544430e+00]
 [-1.02184904e+00 -2.43394714e+00 -1.46640561e-01 -2.62386821e-01]
 [-4.16009689e-01 -1.05276654e+00  3.64896281e-01  8.77547895e-04]
 [ 1.28034050e+00  3.28414053e-01  1.10378283e+00  1.44883158e+00]]


**Step 4**: Always check the ported transformer: the score is the fraction
of samples whose transformed features match those of the original
transformer:

In [5]:
score = transformer.test(X[::15])

print(score)

1.0


The class `Porter` and the functions `make` and `test` accept transformers
too:

In [6]:
from sklearn_porter import Porter, make, test

X_tr = Porter(scaler, language='js').transform(X[::30])
X_tr = make(scaler, X[::30], language='js')
score = test(scaler, X[::15], language='js')

print(score)

1.0


## Use the ported transformer

The ported transformer reads the features of one sample from its arguments
or of several samples (one per line) from the standard input. It prints the
transformed features of each sample in one line (separated by spaces),
which is also the input format of the ported estimators:

In [7]:
!node /tmp/StandardScaler.js 5.1 3.5 1.4 0.2

-0.9006811702978099 1.0190043519716065 -1.3402265266227635 -1.3154442950077407


## Change the number of features

Transformers can change the number of features, e.g. a `KBinsDiscretizer`
with a one-hot encoding of the bins. A `ColumnTransformer` (or a
`FeatureUnion`) combines the transformed features of several transformers,
if the transformers of all its parts are supported:

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import KBinsDiscretizer

columns = ColumnTransformer(
    [
        ('scale', StandardScaler(), [0, 1]),
        ('bins', KBinsDiscretizer(n_bins=3, encode='onehot-dense'), [2, 3]),
    ]
)
columns.fit(X)

transformer = Transformer(columns, language='js')
X_tr = transformer.make(X[::30])

print(X_tr)
print(transformer.test(X[::15]))

[[-0.90068117  1.01900435  1.          0.          0.          1.
   0.          0.        ]
 [-1.26418478  0.09821729  1.          0.          0.          1.
   0.          0.        ]
 [-1.02184904 -2.43394714  0.          1.          0.          0.
   1.          0.        ]
 [-0.41600969 -1.05276654  0.          1.          0.          0.
   1.          0.        ]
 [ 1.2803405   0.32841405  0.          0.          1.          0.
   0.          1.        ]]


1.0


The [README](https://github.com/nok/sklearn-porter#transformers) lists all
supported transformers, the
[estimator notes](https://github.com/nok/sklearn-porter/blob/main/docs/estimator-notes.md)
describe the parameters of single transformers which can't be ported.

## Next steps

- [Pipelines](pipelines.ipynb): port the preprocessing and the prediction
  of a pipeline to one source file.